In [1]:
cd ./wrkDir/s19/s19-1/

/work/xiangchong.li/work/galSim-HSC/s19/s19-1


In [2]:
%pylab inline
import os
import numpy as np

import pandas as pd
import matplotlib
import astropy.io.fits as pyfits
import astropy.io.ascii as ascii
from astropy.table import Table
import matplotlib.pyplot as plt

import sim_analysis_utilities as utilities

# The weight is updated according to the seeing and nosie variance
nbin1       =   20
fwhm_bin    =   np.linspace(0.3,0.9,nbin1)
fwhm_bin_0  =   fwhm_bin[:-1]
fwhm_bin_1  =   fwhm_bin[1:]
gridPsf     =   (fwhm_bin_0+fwhm_bin_1)/2.
var_bin     =   np.linspace(0.001,0.01,nbin1)
var_bin_0   =   var_bin[:-1]
var_bin_1   =   var_bin[1:]
gridNoi     =   (var_bin_0+var_bin_1)/2.


def getObsHist(data):
    fwhms   =   utilities.get_psf_size(data)
    varNois =   utilities.get_noi_var(data)
    histNoi =   np.histogram(varNois,bins=var_bin,density=True)[0]
    histPsf =   np.histogram(fwhms,bins=fwhm_bin,density=True)[0]
    return Table(data=[gridNoi,histNoi,gridPsf,histPsf,len(data)*np.ones(len(gridNoi))],\
                names=disnames)

def getSimHist(data):
    #Get the noise variance and psf information
    mask    =   utilities.get_wl_flags(data)
    data    =   data[mask]
    varNois =   utilities.get_noi_var(data)
    fwhms   =   utilities.get_psf_size(data)
    histNoi =   np.histogram(varNois,bins=var_bin,density=True)[0]
    histPsf =   np.histogram(fwhms,bins=fwhm_bin,density=True)[0]
    return Table(data=[gridNoi,histNoi,gridPsf,histPsf,len(data)*np.ones(len(gridNoi))],names=disnames)

def reWeight(data,t_dis,t_dis_obs):
    varNois =   utilities.get_noi_var(data)
    fwhms   =   utilities.get_psf_size(data)
    #reweight for noise
    ratio_noi=   np.array(t_dis_obs['histNoi']/t_dis['histNoi'])
    weightNoi=  np.zeros(len(data))
    for ii,vn in enumerate(varNois):
        if vn<=var_bin[0] or vn>var_bin[-1]:
            weightNoi[ii]   =   0.
        else:
            ind             =   np.where((vn>var_bin_0) & (vn<var_bin_1))[0][0]
            weightNoi[ii]   =   ratio_noi[ind]
    #apply the noise weight
    histNoi_rn  =   np.histogram(varNois,bins=var_bin,weights=weightNoi,density=True)[0]
    histPsf_rn  =   np.histogram(fwhms,bins=fwhm_bin,weights=weightNoi,density=True)[0]

    #reweight for PSF
    ratios  =   np.array(t_dis_obs['histPsf']/histPsf_rn)
    weightPsf=  np.zeros(len(data))
    for ii,rp in enumerate(fwhms):
        if rp<=fwhm_bin[0] or rp>fwhm_bin[-1]:
            weightPsf[ii] =   0.
        else:
            ind     =   np.where((rp>fwhm_bin_0) & (rp<fwhm_bin_1))[0][0]
            ratio   =   ratios[ind]
            if np.isnan(ratio) or abs(ratio)==np.inf:
                weightPsf[ii]   =   0.
            else:
                weightPsf[ii]   =   ratio
    reweight    =   weightPsf*weightNoi
    data['weight']   =   reweight
    histNoi_rp  =   np.histogram(varNois,bins=var_bin,weights=data['weight'],density=True)[0]
    histPsf_rp  =   np.histogram(fwhms,bins=fwhm_bin,weights=data['weight'],density=True)[0]
    return Table(data=[gridNoi,histNoi_rp,gridPsf,histPsf_rp,len(data)*np.ones(len(gridNoi))],names=disnames)

Populating the interactive namespace from numpy and matplotlib


In [3]:
anaDir0='./anaCat_newS19Mask/'
anaDir1='./anaCat_newS19Mask_fdeltacut/'
obsDir='./anaCat_newS19Mask_fdeltacut/catalog_obs_reGaus_no_m/'

In [4]:
inFname=os.path.join(anaDir0,'allSim.h5')
outFname=os.path.join(anaDir1,'allSim.h5')
allSim=pd.read_hdf(inFname,key='data',mode='a')
info0=ascii.read('info_newS19Mask_sim.csv')
info1=ascii.read('info_newS19Mask_fd001_sim.csv')
print(len(allSim))

19957043


In [5]:
# Drop removed PSFs
if not os.path.isfile(outFname):
    droplist=[nn for nn in info0['num'] if nn not in info1['num']]
    allSim=allSim.drop(droplist,level='isim')
    allSim.to_hdf(outFname,key='data',overwrite=True)
else:
    allSim=pd.read_hdf(outFname,key='data',mode='r')
print(len(allSim))

19542561


In [6]:
# update weight
disnames    =   ('gridNoi','histNoi','gridPsf','histPsf','num')
for fieldname in utilities.field_names:
    anaFieldDir =   os.path.join(anaDir1,'%s' %(fieldname))
    if not os.path.isdir(anaFieldDir):
        os.mkdir(anaFieldDir)
    _fname      =   os.path.join(obsDir,'%s_no_m.fits' %(fieldname))
    dataObs     =   pyfits.getdata(_fname)
    
    outDnameObs =   os.path.join(anaFieldDir,'varPSF_dis_obs.csv')
    t_dis_obs   =   getObsHist(dataObs)
    t_dis_obs.write(outDnameObs,overwrite=True)
    
    dataSim=allSim.xs(fieldname,level='fieldname').to_records()
    outDname    =   os.path.join(anaFieldDir,'varPSF_dis_sim.csv')
    t_dis_sim   =   getSimHist(dataSim)
    t_dis_sim.write(outDname,overwrite=True)
    
    outDnameRw  =   os.path.join(anaFieldDir,'varPSF_dis_rw.csv')
    t_dis_rw    =   reWeight(dataSim,t_dis_sim,t_dis_obs)
    t_dis_rw.write(outDnameRw,overwrite=True)
    idx = pd.IndexSlice
    print(allSim.loc[idx[fieldname,:,:],'weight'][0])
    allSim.loc[idx[fieldname,:,:],'weight']=dataSim['weight']
    print(allSim.loc[idx[fieldname,:,:],'weight'][0])

/work/xiangchong.li/superonionGW/code/FPFS_Private/FPFSBASE/python/sim_analysis_utilities.py:1781: RuntimeWarning: invalid value encountered in less
  (get_imag_A10(catalog)<25.5)& \
/work/xiangchong.li/hscpipe7/python/miniconda3-4.5.12/envs/lsst-scipipe-1172c30/lib/python3.7/site-packages/ipykernel_launcher.py:48: RuntimeWarning: divide by zero encountered in true_divide
/work/xiangchong.li/hscpipe7/python/miniconda3-4.5.12/envs/lsst-scipipe-1172c30/lib/python3.7/site-packages/ipykernel_launcher.py:48: RuntimeWarning: invalid value encountered in true_divide
/work/xiangchong.li/hscpipe7/python/miniconda3-4.5.12/envs/lsst-scipipe-1172c30/lib/python3.7/site-packages/ipykernel_launcher.py:61: RuntimeWarning: divide by zero encountered in true_divide
/work/xiangchong.li/hscpipe7/python/miniconda3-4.5.12/envs/lsst-scipipe-1172c30/lib/python3.7/site-packages/ipykernel_launcher.py:61: RuntimeWarning: invalid value encountered in true_divide


0.7804463092248975


/work/xiangchong.li/hscpipe7/python/miniconda3-4.5.12/envs/lsst-scipipe-1172c30/lib/python3.7/site-packages/IPython/core/interactiveshell.py:2848: PerformanceWarning: indexing past lexsort depth may impact performance.
  raw_cell, store_history, silent, shell_futures)
/work/xiangchong.li/hscpipe7/python/miniconda3-4.5.12/envs/lsst-scipipe-1172c30/lib/python3.7/site-packages/IPython/core/async_helpers.py:68: PerformanceWarning: indexing past lexsort depth may impact performance.
  coro.send(None)


0.7618176113984801
0.6238042747554134
0.6599515625449404
0.631950863739234
0.6322659039228815
1.0418641408486315
1.039538386370403
0.9573722428918854
0.953241813058599
1.117228857971013
1.1225857932789027


In [10]:
allSim.to_hdf(outFname,key='data',overwrite=True)

In [12]:
for fieldname in utilities.field_names:
    outDir=os.path.join(anaDir1,fieldname,'smallcats')
    if not os.path.exists(outDir):
        os.mkdir(outDir)
    datField=allSim.xs(fieldname,level='fieldname')
    isimList=info1[info1['field']==fieldname]['num']
    print(fieldname)
    for isim in isimList:
        if not isim in allSim.index.levels[1]:
            print(isim)
            continue
        outFname=os.path.join(outDir,'sim-%05d.fits' %isim)
        data=datField.xs(isim,level='isim').to_records(index=True)
        pyfits.writeto(outFname,data,overwrite=True)

XMM
1866
GAMA09H
WIDE12H
1666
GAMA15H
VVDS
HECTOMAP
